In [1]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os


/home/mouad/anaconda3/envs/rag-assistant/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Extraction du texte du PDF
pdf_filename = os.listdir("../data")[0]
doc = fitz.open(f"../data/{pdf_filename}")

text = ""
for page in doc:
    text += page.get_text()
doc.close()

print(f"Longueur du texte extrait : {len(text)} caractères")
print("\n--- Aperçu ---")
print(text[:500])

Longueur du texte extrait : 91859 caractères

--- Aperçu ---
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
CLOUD COMPUTING ET VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
Master  Ingénierie Informatique et Sécurité Systèmes 
OBJECTIFS
Maîtriser les technologies de virtualisation et leurs applications
Comprendre et maîtriser les modèles de cloud computing
Savoir déployer et administrer des infrastructures virtualisées (hyperviseurs, conteneurs, 
stockage réseau).
Maîtriser les différents modèles de déploiement cloud (public, privé, hybride, comm


In [3]:
# 2. Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_text(text)

print(f"\nNombre de chunks : {len(chunks)}")
print("\n--- Chunk 1 ---")
print(chunks[0])
print("\n--- Chunk 2 ---")
print(chunks[1])


Nombre de chunks : 204

--- Chunk 1 ---
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
CLOUD COMPUTING ET VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
Master  Ingénierie Informatique et Sécurité Systèmes 
OBJECTIFS
Maîtriser les technologies de virtualisation et leurs applications
Comprendre et maîtriser les modèles de cloud computing
Savoir déployer et administrer des infrastructures virtualisées (hyperviseurs, conteneurs, 
stockage réseau).

--- Chunk 2 ---
stockage réseau).
Maîtriser les différents modèles de déploiement cloud (public, privé, hybride, communautaire).
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
Chp1 : VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
INTRODUCTION A LA VIRTUALISATION
une entreprise veut une architecture 3-tier : 
1 serveur web
1 serveur d'application
1 serveur de base de données
Comment déployer tout ça sur du VRAI matériel sans se ruiner ?
SOLUTION NAÏVE :
Acheter 3 serveurs physiques →15 000€


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3030.42it/s]


In [7]:
import chromadb
import hashlib

client_db = chromadb.PersistentClient(path="../vectorstore")
collection = client_db.get_or_create_collection(name="mon_cours")


# on génère des ids basés sur le hash du contenu (au lieu de chunk_0, chunk_1...)
ids = [hashlib.md5(chunk.encode()).hexdigest() for chunk in chunks]

embeddings = model.encode(chunks).tolist()

collection.upsert(
    documents=chunks,
    embeddings=embeddings,
    ids=ids
)

print(f"{len(chunks)} chunks ajoutés/mis à jour dans ChromaDB (total dans la collection : {collection.count()})")

204 chunks ajoutés/mis à jour dans ChromaDB (total dans la collection : 204)


In [8]:
question = "pourquoi le cloud computing est-il important ?"  # ou une vraie question sur ton PDF

# Étape 1 : encoder la question EXACTEMENT comme   les chunks
question_embedding = model.encode([question]).tolist()  # liste car encode() attend une liste

# Étape 2 : query ChromaDB avec cet embedding
results = collection.query(
    query_embeddings=question_embedding,
    n_results=3  # top 3 chunks les plus proches
)

# Étape 3 : afficher les résultats
for i, doc in enumerate(results["documents"][0]):
    print(f"--- Résultat {i+1} ---")
    print(doc)
    print()

--- Résultat 1 ---
permet
aux
utilisateurs
et
aux
entreprises
de
se
libérer de la nécessité de gérer des
serveurs
physiques
eux-mêmes
ou
d'exécuter des applications logicielles sur
leurs propres équipements.
•
Définition du Cloud Computing
•
Le cloud est une métaphore pour désigner Internet. Il représente une
abstraction
d'Internet
et
de
l'infrastructure
qui
le
soutient.
Le
terme computing fait référence aux services informatiques fournis par un

--- Résultat 2 ---
la réduction du coût total de possession des systèmes informatiques, la
facilité d'augmenter ou de diminuer les ressources.
•
Le cloud computing peut permettre d'effectuer des économies, notamment
grâce à la mutualisation des services sur un grand nombre de clients. Certains
analystes indiquent que 20 à 25 % d’économies pourraient être réalisées par
les gouvernements sur leur budget informatique s’ils migraient vers le cloud
computing33.
•

--- Résultat 3 ---
Les caractéristiques du Cloud Computing
Buy
1. Le Service à la dem

In [9]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # charge le .env (project root, donc si t'es dans notebooks/ faut peut-être load_dotenv("../.env"))

client_groq = Groq(api_key=os.getenv("GROQ_API_KEY")) 

In [10]:
# on récupère les chunks les plus pertinents (déjà fait au-dessus avec results)
context = "\n\n".join(results["documents"][0])

prompt = f"""Tu es un assistant qui répond aux questions en te basant UNIQUEMENT sur le contexte fourni ci-dessous, extrait d'un cours.

Contexte :
{context}

Question : {question}

Réponds de manière claire et concise, en te basant uniquement sur le contexte. Si l'information n'est pas dans le contexte, dis-le clairement plutôt que d'inventer."""

response = client_groq.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2  # bas = réponses plus factuelles, moins créatives
)

answer = response.choices[0].message.content
print(answer)

Selon le contexte fourni, le cloud computing est important car il permet de réaliser des économies, notamment grâce à la mutualisation des services sur un grand nombre de clients. En effet, certains analystes indiquent que 20 à 25 % d’économies pourraient être réalisées par les gouvernements sur leur budget informatique s’ils migraient vers le cloud computing.
